<a href="https://colab.research.google.com/github/Flora-M04/WBC_detection/blob/main/notebooks/yolov11_bccd_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics -q
import requests
from ultralytics import YOLO
import os
from PIL import Image # Added missing import

# 1. Download the image first
url = "https://ultralytics.com/images/bus.jpg"
response = requests.get(url)
with open("image.jpg", "wb") as f:
   f.write(response.content)

# 2. Verify file exists and is valid
if os.path.exists("image.jpg"):
    try:
        img = Image.open("image.jpg")
        img.verify() # Verify if it's a valid image
        print("Image downloaded successfully and is valid. Running inference...")
        # 3. Load the model
        model = YOLO("yolo11n.pt")

        # 4. Run prediction
        results = model("image.jpg")

        # 5. Visualize the results
        for r in results:
            im_array = r.plot()  # plot a BGR numpy array of predictions
            im = Image.fromarray(im_array[..., ::-1])  # RGB PIL image
            display(im)
    except Exception as e:
        print(f"Error: Downloaded image is corrupted or invalid. {e}")
else:
    print("Error: Failed to download the image.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.9 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import drive
from ultralytics import YOLO

# 1. Mount Drive
drive.mount('/content/drive', force_remount=True)

# 2. Define Paths
BASE_PATH = '/content/drive/MyDrive/wbc_yolo'
os.makedirs(BASE_PATH, exist_ok=True)

# 3. Search for the weights file
search_path = os.path.join(BASE_PATH, 'exports')
print("\nSearching for your trained models in:", search_path)

if os.path.exists(search_path):
    for root, dirs, files in os.walk(search_path):
        if 'best.pt' in files:
            print("\n FOUND IT! Copy this exact path:")
            print(f"{os.path.join(root, 'best.pt')}")
else:
    print(" Could not find the exports folder. Did you save it somewhere else?")

# 4. Download the weights (yolo11n.pt)
model = YOLO('yolo11n.pt')
model.save(os.path.join(BASE_PATH, 'yolo11n.pt'))

# 5. Generate the wbc_data.yaml
yaml_content = f"""
path: /content/WBC_Dataset
train: images/train
val: images/val

nc: 1
names: ['WBC']
"""
with open(os.path.join(BASE_PATH, 'wbc_data.yaml'), 'w') as f:
    f.write(yaml_content)

print(" Folder structure and support files are ready!")

Mounted at /content/drive

Searching for your trained models in: /content/drive/MyDrive/wbc_yolo/exports

 FOUND IT! Copy this exact path:
/content/drive/MyDrive/wbc_yolo/exports/wbc_detection_model/weights/best.pt

 FOUND IT! Copy this exact path:
/content/drive/MyDrive/wbc_yolo/exports/wbc_detection_model-2/weights/best.pt
 Folder structure and support files are ready!


In [ ]:
import os

# 1. Write the streamlit code to a file
with open('app.py', 'w') as f:
    f.write('''
import streamlit as st
import torch
from ultralytics import YOLO
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

st.set_page_config(page_title="WBC POCT System", layout="wide")
st.title("AI-Based Portable WBC Classification System")

PX_TO_UM_RATIO = 0.5
T1 = 9.0
T2 = 15.0

@st.cache_resource
def load_model():
    return YOLO("yolo11n.pt")

model = load_model()

st.sidebar.header("System Controls")
uploaded_file = st.sidebar.file_uploader("Insert Test Card Image...", type=["jpg", "jpeg", "png"])
confidence = st.sidebar.slider("Confidence Threshold", 0.1, 1.0, 0.4)

if uploaded_file is not None:
    file_bytes = np.asarray(bytearray(uploaded_file.read()), dtype=np.uint8)
    image = cv2.imdecode(file_bytes, 1)

    with st.spinner("Scanning Blood Sample..."):
        results = model(image, conf=confidence)[0]
        counts = {"LYM": 0, "NEU": 0, "MON": 0}
        all_diameters = []

        for box in results.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            pixel_diameter = max(x2 - x1, y2 - y1)
            um_diameter = pixel_diameter * PX_TO_UM_RATIO
            all_diameters.append(um_diameter)
            if um_diameter < T1: counts["LYM"] += 1
            elif T1 <= um_diameter < T2: counts["NEU"] += 1
            else: counts["MON"] += 1

    # 3. DISPLAY GUI LAYOUT (Corrected Indentation Starts Here)
    col1, col2 = st.columns([2, 1])

    with col1:
        st.subheader("Automated Microscopic Detection")
        annotated_img = results.plot()
        st.image(cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB), use_column_width=True)

    with col2:
      st.subheader("Diagnostic Report")
    total = sum(counts.values())
    st.metric("Total WBC Count", total)
    for cell, count in counts.items():
     perc = (count / total * 100) if total > 0 else 0
     st.write(f"**{cell}:** {count} ({perc:.1f}%)")

    # Diameter Histogram (Day 4 Preview)
    fig, ax = plt.subplots()
    ax.hist(all_diameters, bins=10, color='skyblue', edgecolor='black')
    ax.set_title("WBC Diameter Distribution (µm)")
    ax.set_xlabel("Diameter")
    st.pyplot(fig)
else:
    st.info("Please upload a blood sample image to start.")
''')
 # 2. Install localtunnel
!npm install -g localtunnel -q

# 3. Print the Tunnel Password (required for localtunnel)
import urllib
print("Your Tunnel Password is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

# 4. Run Streamlit in the background and start the tunnel
get_ipython().system_raw('streamlit run app.py & npx localtunnel --port 8501 &')
print("\n1. Click the link ending in .loca.lt below when it appears.")
print("2. Paste the Tunnel Password from above into the 'Endpoint IP' field on that page.")


⠙⠹⠸⠼⠴⠦⠧
changed 22 packages in 992ms
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧Your Tunnel Password is: 136.109.187.184

1. Click the link ending in .loca.lt below when it appears.
2. Paste the Tunnel Password from above into the 'Endpoint IP' field on that page.


In [ ]:

import xml.etree.ElementTree as ET
import shutil
import zipfile
import os
from sklearn.model_selection import train_test_split

# --- STEP 1: CONFIGURATION ---
XML_ZIP = '/content/drive/MyDrive/wbc_yolo/BCCD_Dataset-master.zip'
DRIVE_IMG_DIR = '/content/drive/MyDrive/wbc_yolo/JPEGImages'
OUTPUT_BASE = '/content/WBC_Dataset'
TMP_EXTRACT = '/content/bccd_temp'

# --- STEP 2: UNZIP & AUTO-LOCATE ---
if not os.path.exists(TMP_EXTRACT):
    with zipfile.ZipFile(XML_ZIP, 'r') as zip_ref:
        zip_ref.extractall(TMP_EXTRACT)

# Diagnostic: Let's see what is actually inside the ZIP
print("Searching for folders in:", TMP_EXTRACT)
XML_SOURCE = None
IMG_SOURCE = DRIVE_IMG_DIR if os.path.exists(DRIVE_IMG_DIR) else None

for root, dirs, files in os.walk(TMP_EXTRACT):
    # Case-insensitive check for common names
    if any(d.lower() == 'annotations' or d.lower() == 'annotation' for d in dirs):
        XML_SOURCE = os.path.join(root, next(d for d in dirs if d.lower() in ['annotations', 'annotation']))
    if not IMG_SOURCE and any(d.lower() == 'jpegimages' for d in dirs):
        IMG_SOURCE = os.path.join(root, next(d for d in dirs if d.lower() == 'jpegimages'))

if not XML_SOURCE:
    print("Full directory tree found:")
    for root, dirs, files in os.walk(TMP_EXTRACT):
        print(f"    {root}")
    raise FileNotFoundError(f"Could not find an 'Annotation' folder in {TMP_EXTRACT}")

print(f"Found XML Source: {XML_SOURCE}")
print(f"Found Image Source: {IMG_SOURCE}")

# --- STEP 3: CREATE DIRECTORIES ---
for folder in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    os.makedirs(os.path.join(OUTPUT_BASE, folder), exist_ok=True)

# --- STEP 4: YOLO CONVERSION ---
def convert_to_yolo(xml_file, output_txt):
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        size = root.find('size')
        w, h = int(size.find('width').text), int(size.find('height').text)

        found_wbc = False
        with open(output_txt, 'w') as f:
            for obj in root.iter('object'):
                if obj.find('name').text == 'WBC':
                    found_wbc = True
                    box = obj.find('bndbox')
                    xn, yn = float(box.find('xmin').text), float(box.find('ymin').text)
                    xx, yx = float(box.find('xmax').text), float(box.find('ymax').text)
                    f.write(f"0 {(xn+xx)/2/w} {(yn+yx)/2/h} {(xx-xn)/w} {(yx-yn)/h}\n")
        return found_wbc
    except Exception as e:
        return False

# --- STEP 5: SPLIT & FINALIZE ---
all_xmls = [f for f in os.listdir(XML_SOURCE) if f.endswith('.xml')]
if not all_xmls:
    print(f"Warning: No .xml files found in {XML_SOURCE}")
else:
    train_xmls, val_xmls = train_test_split(all_xmls, test_size=0.2, random_state=42)

    def process_data(file_list, split):
        count = 0
        for xml_name in file_list:
            base = xml_name.replace('.xml', '')
            xml_p, img_p = os.path.join(XML_SOURCE, xml_name), os.path.join(IMG_SOURCE, base + '.jpg')
            if os.path.exists(img_p):
                txt_p = os.path.join(OUTPUT_BASE, 'labels', split, base + '.txt')
                if convert_to_yolo(xml_p, txt_p):
                    shutil.copy(img_p, os.path.join(OUTPUT_BASE, 'images', split, base + '.jpg'))
                    count += 1
        print(f" {split.capitalize()}: {count} images ready.")

    process_data(train_xmls, 'train')
    process_data(val_xmls, 'val')
# Create the yaml file in your current Colab directory
yaml_content = f"""
path: /content/WBC_Dataset
train: images/train
val: images/val

nc: 1
names: ['WBC']
"""

with open('/content/wbc_data.yaml', 'w') as f:
    f.write(yaml_content)
print("wbc_data.yaml is ready.")
from ultralytics import YOLO

# 1. Load the pre-trained nano model (Backbone for transfer learning)
model = YOLO('yolo11n.pt')

# 2. Start Training
# epochs=50: Enough for BCCD to converge
# imgsz=640: Standard resolution for clear cell detection
results = model.train(
    data='/content/wbc_data.yaml',
    epochs=50,
    imgsz=640,
    plots=True,
    project='/content/drive/MyDrive/wbc_yolo/exports', # Saves results to your Drive
    name='wbc_detection_model'
)


Searching for folders in: /content/bccd_temp
Found XML Source: /content/bccd_temp/BCCD_Dataset-master/BCCD/Annotations
Found Image Source: /content/drive/MyDrive/wbc_yolo/JPEGImages
 Train: 287 images ready.
 Val: 71 images ready.
wbc_data.yaml is ready.
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/wbc_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_widt

In [ ]:
%%writefile app.py
import streamlit as st
from ultralytics import YOLO
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

st.set_page_config(page_title="WBC POCT System", layout="wide")
st.title("🔬 AI-Based Portable WBC Classification System")

@st.cache_resource
def load_model():
    return YOLO('/content/drive/MyDrive/wbc_yolo/exports/wbc_detection_model/weights/best.pt')

try:
    model = load_model()
except Exception as e:
    st.error("Model weights not found. Check your file path!")
    st.stop()

# --- Dynamic Controls ---
st.sidebar.header("System Controls")
st.sidebar.markdown("Adjust the Ratio if Monocytes are being misclassified.")
RATIO = st.sidebar.number_input("Calibration Ratio (µm/px)", value=0.0156, format="%.4f", step=0.0001)
T1 = st.sidebar.number_input("LYM/NEU Threshold (µm)", value=9.0, step=0.5)
T2 = st.sidebar.number_input("NEU/MON Threshold (µm)", value=14.5, step=0.5)
CONFIDENCE = st.sidebar.slider("Confidence Threshold", 0.1, 1.0, 0.25)
MARGIN = 5

uploaded_file = st.sidebar.file_uploader("Upload Blood Smear Image", type=["jpg", "jpeg", "png", "bmp"])

if uploaded_file is not None:
    file_bytes = np.asarray(bytearray(uploaded_file.read()), dtype=np.uint8)
    image = cv2.imdecode(file_bytes, 1)
    img_h, img_w = image.shape[:2]

    with st.spinner("Scanning Blood Sample..."):
        results = model(image, conf=CONFIDENCE)[0]
        counts = {"LYM": 0, "NEU": 0, "MON": 0}
        all_diameters = []

        for box in results.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            w, h = x2 - x1, y2 - y1
            longer_side = max(w, h)
            shorter_side = min(w, h)

            is_at_edge = (x1 <= MARGIN) or (y1 <= MARGIN) or (x2 >= img_w - MARGIN) or (y2 >= img_h - MARGIN)
            if is_at_edge and (shorter_side < 0.5 * longer_side):
                continue

            um_diameter = longer_side * RATIO
            all_diameters.append(um_diameter)

            if um_diameter < T1:
                counts["LYM"] += 1
            elif T1 <= um_diameter < T2:
                counts["NEU"] += 1
            else:
                counts["MON"] += 1

        col1, col2 = st.columns([2, 1])

        with col1:
            st.subheader("Microscopic Detection Results")
            annotated_img = results.plot()
            st.image(cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB), use_container_width=True)

        with col2:
            st.subheader("Diagnostic Report")
            total = sum(counts.values())
            st.metric("Total WBC Count", total)

            if total > 0:
                for cell, count in counts.items():
                    perc = (count / total * 100)
                    st.write(f"**{cell}:** {count} ({perc:.1f}%)")

                fig, ax = plt.subplots(figsize=(5,4))
                ax.hist(all_diameters, bins=10, color='skyblue', edgecolor='black')
                ax.axvline(T1, color='red', linestyle='--', label='T1')
                ax.axvline(T2, color='green', linestyle='--', label='T2')

                # Force Y-axis to show only integers
                ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

                ax.set_title("WBC Diameter Distribution (µm)")
                ax.set_xlabel("Diameter (µm)")
                ax.set_ylabel("Count (Frequency)")
                st.pyplot(fig)
else:
    st.info("Awaiting Test Card... Please upload an image.")

Overwriting app.py


In [ ]:
import urllib.request
import subprocess
import time
import os
import re
import sys

# 1. Clean up the environment
os.system("pkill -f streamlit")
os.system("pkill -f cloudflared")

print("Verifying dependencies...")
os.system(f"{sys.executable} -m pip install streamlit ultralytics opencv-python pillow matplotlib -q")

if not os.path.exists("cloudflared"):
    print("Downloading secure tunnel...")
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "cloudflared")
    os.chmod("cloudflared", 0o777)

# 2. Start Streamlit and capture logs
print("Booting up Streamlit on port 8501...")
os.system(f"nohup {sys.executable} -m streamlit run app.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &")

# Give it time to boot or crash
time.sleep(6)

# 3. DIAGNOSTIC CHECK: Did Streamlit crash?
with open("streamlit.log", "r") as f:
    log_content = f.read()

if "Traceback" in log_content or "Error:" in log_content or "ModuleNotFoundError" in log_content:
    print("\n STREAMLIT CRASHED IN THE BACKGROUND! ")
    print("Here is the exact error causing the tunnel to fail:\n")
    print("-" * 40)
    print(log_content)
    print("-" * 40)
else:
    print(" Streamlit is running smoothly!")
    print(" Generating your public link...")

    # 4. Start Tunnel
    process = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8501"],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

    for line in iter(process.stdout.readline, ''):
        if "trycloudflare.com" in line:
            url_match = re.search(r'(https://[a-zA-Z0-9-]+\.trycloudflare\.com)', line)
            if url_match:
                print("\n" + "="*60)
                print("SUCCESS! THE LINK IS READY. ")
                print(" IMPORTANT: Count to 10 slowly before clicking it to allow DNS to update!")
                print(f"\n{url_match.group(1)}\n")
                print("="*60)
                print("(Leave this cell running while you test your app!)")
                break

Verifying dependencies...
Booting up Streamlit on port 8501...
 Streamlit is running smoothly!
 Generating your public link...

SUCCESS! THE LINK IS READY. 
 IMPORTANT: Count to 10 slowly before clicking it to allow DNS to update!

https://xhtml-likely-assess-big.trycloudflare.com

(Leave this cell running while you test your app!)


In [ ]:
# 1. Re-install streamlit
!pip install streamlit -q

# 2. Install localtunnel
!npm install -g localtunnel -q

# 3. Run Streamlit in the background and start the tunnel
# get_ipython() is a global in Colab; no need to import it.
get_ipython().system_raw('streamlit run app.py & npx localtunnel --port 8501 &')

print("Streamlit is starting in the background...")
print("Please wait for the tunnel link to appear in the console output if applicable, or check the previous instruction for the URL.")

⠙⠹⠸⠼⠴
changed 22 packages in 671ms
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴Streamlit is starting in the background...
Please wait for the tunnel link to appear in the console output if applicable, or check the previous instruction for the URL.
